In [ ]:
from loka.data import load_breakthrough_states
from loka.geometrical_methods import k_center_order
from loka.embeddings import with_counts, with_near_goal, with_counts_and_near_goal
import numpy as np
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
X,Y = load_breakthrough_states("/home/axelbjarkar/RU/T-404-LOKA-code/data/e.txt", two_channel=False)

In [ ]:
X = with_counts(X)

In [ ]:
order = k_center_order(X, 0.2, device, S0='rand')

In [ ]:
order

In [ ]:
# Convert to torch tensors on device if they aren't already
X_tensor = torch.as_tensor(X, dtype=torch.float32, device=device)
Y_tensor = torch.as_tensor(Y, device=device)

# Flatten feature dimensions if X has image/grid shapes like (N, C, H, W)
if X_tensor.ndim > 2:
    X_flat = X_tensor.flatten(start_dim=1)
else:
    X_flat = X_tensor

# Get the center representations
centers = X_flat[order]  # Shape: (num_centers, feature_dim)

# Process in batches to prevent CUDA Out-Of-Memory on large datasets
batch_size = 4096
closest_centers = []

for i in range(0, len(X_flat), batch_size):
    batch = X_flat[i : i + batch_size]
    # Compute pairwise Euclidean distances between batch and centers: (B, num_centers)
    dists = torch.cdist(batch, centers)
    
    # Index of the closest center in the `order` array
    min_idx_in_order = torch.argmin(dists, dim=1)
    
    # Map back to the original index in X / Y
    closest_center_idx = torch.as_tensor(order, device=device)[min_idx_in_order]
    closest_centers.append(closest_center_idx)

# Concatenate results for all x in X
closest_f = torch.cat(closest_centers)

# Check label agreement: Y[x] == Y[f]
matches = (Y_tensor == Y_tensor[closest_f])

# Summary statistics
accuracy = matches.float().mean().item()
print(f"Nearest center label accuracy: {accuracy * 100:.2f}%")

In [ ]:
matches

In [ ]:
import pandas as pd
import torch

# 1. Prepare data
X_tensor = torch.as_tensor(X, dtype=torch.float32, device=device)
Y_tensor = torch.as_tensor(Y, device=device)

if X_tensor.ndim > 2:
    X_flat = X_tensor.flatten(start_dim=1)
else:
    X_flat = X_tensor

order_tensor = torch.as_tensor(order, device=device)
centers = X_flat[order_tensor]  # Shape: (K, D)

# 2. Find the closest facility and distance for every x in X
batch_size = 4096
closest_facility_indices = []
min_distances = []

for i in range(0, len(X_flat), batch_size):
    batch = X_flat[i : i + batch_size]
    dists = torch.cdist(batch, centers)  # (batch_size, num_facilities)
    min_dist, min_idx = torch.min(dists, dim=1)

    closest_facility_indices.append(min_idx)
    min_distances.append(min_dist)

# Index in `order` (0 to len(order)-1)
assigned_facility_pos = torch.cat(closest_facility_indices)
assigned_distances = torch.cat(min_distances)

# Original facility index in X
assigned_f = order_tensor[assigned_facility_pos]

# Check if label matches
matches = Y_tensor == Y_tensor[assigned_f]

# 3. Aggregate statistics for each facility
facility_data = []

for pos, f_idx in enumerate(order):
    mask = assigned_facility_pos == pos
    num_assigned = mask.sum().item()

    if num_assigned > 0:
        assigned_pts = torch.where(mask)[0].cpu().numpy()
        dists_f = assigned_distances[mask]
        match_count = matches[mask].sum().item()
        accuracy = match_count / num_assigned
        mean_dist = dists_f.mean().item()
        max_dist = dists_f.max().item()
    else:
        assigned_pts = []
        match_count = 0
        accuracy = 0.0
        mean_dist = 0.0
        max_dist = 0.0

    facility_data.append(
        {
            "facility_order_idx": pos,
            "facility_X_idx": int(f_idx),
            "label": Y[f_idx].item() if hasattr(Y[f_idx], "item") else Y[f_idx],
            "count": num_assigned,
            "correct_labels": match_count,
            "accuracy": accuracy,
            "mean_dist": mean_dist,
            "max_dist": max_dist,
            "assigned_points": assigned_pts,
        }
    )

df_facilities = pd.DataFrame(facility_data)
df_facilities